In [ ]:
import pandas as pd, numpy as np
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib

In [ ]:
location = "India"
vehicle = "rice"

In [ ]:
location = location.title()

In [ ]:
pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

In [ ]:
asfr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.age_specific_fertility_rate, "estimate", location
).value
asfr[asfr > 0]

In [ ]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
asfr

In [ ]:
sbr = vivarium_inputs.get_measure(
    # GBD 2023 split this into 20-week and 28-week variants; using 28 weeks to
    # match data_keys.SBR and 0500. See the note there -- this is NOT the
    # continuity-preserving choice and cuts the ratio ~32% vs GBD 2021.
    gbd_mapping.covariates.stillbirth_28_weeks_to_live_birth_ratio, "estimate", location
).value
sbr[sbr > 0]

In [ ]:
sbr = sbr.iloc[0]
sbr

In [ ]:
maternal_abortion_miscarriage_incidence = vivarium_inputs.get_measure(
    gbd_mapping.causes.maternal_abortion_and_miscarriage,
    "incidence_rate",
    location,
).mean(axis=1)
maternal_abortion_miscarriage_incidence[maternal_abortion_miscarriage_incidence > 0]

In [ ]:
ectopic_pregnancy_incidence = vivarium_inputs.get_measure(
    gbd_mapping.causes.ectopic_pregnancy,
    "incidence_rate",
    location,
).mean(axis=1)
ectopic_pregnancy_incidence[ectopic_pregnancy_incidence > 0]

In [ ]:
pregnancy_incidence = (
    asfr
    + (asfr * sbr)
    + maternal_abortion_miscarriage_incidence
    + ectopic_pregnancy_incidence
)
pregnancy_incidence[pregnancy_incidence > 0]

In [ ]:
pregnancies = pop * pregnancy_incidence
pregnancies[pregnancies > 0]

In [ ]:
total_pregnancies = pregnancies.sum()
total_pregnancies

In [ ]:
sim_pregnancy_outcomes = pd.read_parquet(
    f"../../0200_pregnancy_sim/sim_results/{vehicle}/{location.lower()}/pregnancy_outcome_count.parquet"
)
n_random_seeds = sim_pregnancy_outcomes.random_seed.nunique()
sim_population = (
    sim_pregnancy_outcomes.groupby(["input_draw", "scenario"]).value.sum().mean()
)
sim_population

In [ ]:
scalar = total_pregnancies / sim_population
scalar

In [ ]:
# `scalar` multiplies every pregnancy number this notebook writes, and until now nothing
# bounded it -- it was displayed and then applied. A zero simulated denominator makes it
# `inf`, which propagates silently into every rescaled parquet rather than failing.
assert np.isfinite(scalar) and scalar > 0, (
    f"rescaling factor is not a positive finite number: {scalar}. "
    f"GBD pregnancies={total_pregnancies:,.0f}, simulated pregnancy outcomes="
    f"{sim_population:,.0f} over {n_random_seeds} seeds. Every parquet below is "
    "multiplied by this."
)

# The scalar itself cannot be banded: it scales as 1/n_random_seeds, so it was 1.06-3.10
# for the published 200-seed run and 32.5-48.1 for a 10-seed one. What *is* seed
# independent is the simulated cohort size per seed. The band leaves ~25x margin either
# side of what both GBD vintages produce; its job is to catch a collapsed or missing
# denominator, not to pin epidemiology.
outcomes_per_seed = sim_population / n_random_seeds
assert 1_000 <= outcomes_per_seed <= 1_000_000, (
    f"{outcomes_per_seed:,.0f} simulated pregnancy outcomes per seed is outside the "
    f"plausible 1,000-1,000,000 range ({sim_population:,.0f} over {n_random_seeds} "
    "seeds). Either the simulation produced almost nothing, or "
    "pregnancy_outcome_count.parquet is not what this notebook expects."
)
f"scalar={scalar:,.4f} from {outcomes_per_seed:,.0f} outcomes/seed x {n_random_seeds} seeds"

In [ ]:
for result in [
    "ylds",
    "ylls",
    "pregnancy_outcome_count",
    "person_time_anemia",
    "transition_count_maternal_disorders",
]:
    df = pd.read_parquet(
        f"../../0200_pregnancy_sim/sim_results/{vehicle}/{location.lower()}/{result}.parquet"
    )
    df.value *= scalar
    path = pathlib.Path(
        f"../results/rescaled_pregnancy_results/{vehicle}/{location.lower()}/{result}.parquet"
    )
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)